In [8]:
# Import standard libraries
from pathlib import Path
import time

# Third-party libraries
from balance_bot_env import BalanceBotEnv
from gymnasium.utils.env_checker import check_env
import numpy as np

In [2]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42

In [3]:
# Initialize environment
env = BalanceBotEnv(mjcf_path=MJCF_PATH, render_mode="human")

# Print observation and action spaces
print(env.observation_space)
print(env.action_space)

Box([ -3.1415927 -20.        -50.        -50.       ], [ 3.1415927 20.        50.        50.       ], (4,), float32)
Box(-1.0, 1.0, (2,), float32)


In [4]:
# Use gymnasium's built-in environment checker
check_env(env)

/opt/rl-env/lib/python3.12/site-packages/gymnasium/utils/env_checker.py:434: UserWarning: WARN: Not able to test alternative render modes due to the environment not having a spec. Try instantiating the environment through `gymnasium.make`
  logger.warn(


In [5]:
# Reset environment and print observation
obs, info = env.reset(seed=SEED)
print(obs)

[0.00073798 0.27395606 0.         0.        ]


In [6]:
# Manual step with no action (don't turn the motors)
obs, reward, terminated, truncated, info = env.step(np.array([0.0, 0.0]))
print(f"obs: {obs}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")

obs: [0.00146857 0.27395606 0.         0.        ]
reward: 0.9999892115592957
terminated: False
truncated: False


In [32]:
# Test: random policy
obs, _ = env.reset()
total_reward = 0

# Go for 200 steps
terminated = False
terminated = False
for step in range(200):
    step_start = time.time()

    # Choose a random action
    action = env.action_space.sample()

    # Take a step with the given action
    obs, reward, terminated, truncated, _ = env.step(action)

    # Accumulate reward
    total_reward += reward

    # Render in MuJoCo
    env.render()

    # Show that we stopped
    if terminated or truncated:
        print(f"Episode ended at step {step}")
        break

    # Make sure the loop iteration time matches the MJCF timestep time (2 ms)
    # i.e. slow the simulation down so we can see the robot behave in real time
    slack = env.model.opt.timestep - (time.time() - step_start)
    if slack > 0:
        time.sleep(slack)

print(f"Final obs: {obs}")
print(f"Final reward: {total_reward}")

Episode ended at step 134
Final obs: [-0.52495885 -4.1743197  -2.7311196   0.87372845]
Final reward: 94.72704315185547


In [33]:
# Close the environment
env.close()